## Unsupervised Anomaly Detection
This notebook ranks unusual tours within each final K-Means cluster. It uses the selected four standardized features and compares flagged tours with the existing quality-review indicators.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

repo_root = Path.cwd()

if not (
    repo_root / "data/processed/tour_level_pca/tour_level_pca_standardized_features.csv"
).exists():
    repo_root = repo_root.parent

features_path = (
    repo_root
    / "data/processed/tour_level_pca/tour_level_pca_standardized_features.csv"
)
assignments_path = (
    repo_root
    / "data/processed/tour_level_pca/tour_level_pca_kmeans_k4_assignments.csv"
)

assert features_path.exists(), f"Feature file not found: {features_path}"
assert assignments_path.exists(), f"Assignment file not found: {assignments_path}"

features_df = pd.read_csv(features_path)
assignments_df = pd.read_csv(assignments_path)

print("Features:", features_df.shape)
print("Assignments:", assignments_df.shape)
print("Feature columns:", features_df.columns.tolist())
print("Assignment columns:", assignments_df.columns.tolist())

### Recreate the selected K-Means clusters

Fit K-Means with \(k=4\) on distance, recorded time, elapsed span, and track count. Keep average speed and quality flags in the tour table for interpreting anomalies, but exclude them from the clustering inputs.

In [ ]:
from sklearn.cluster import KMeans

anomaly_feature_cols = [
    "scaled_total_distance_km",
    "scaled_recorded_track_hours",
    "scaled_elapsed_span_hours",
    "scaled_track_count",
]

X_anomaly = features_df[anomaly_feature_cols].to_numpy(dtype=np.float32)
assert np.isfinite(X_anomaly).all(), "Anomaly-model input has non-finite values"

anomaly_kmeans_model = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=20
)
anomaly_cluster_labels = anomaly_kmeans_model.fit_predict(X_anomaly)

cluster_key_df = features_df[["vehicle_id", "tour_id"]].copy()
cluster_key_df["kmeans_cluster"] = anomaly_cluster_labels

anomaly_data = assignments_df.merge(
    cluster_key_df,
    on=["vehicle_id", "tour_id"],
    how="left",
    validate="one_to_one"
)

print("Tours without a cluster:", int(anomaly_data["kmeans_cluster"].isna().sum()))

print("Cluster counts and existing review flags:")
display(
    anomaly_data.groupby("kmeans_cluster").agg(
        tour_count=("tour_id", "size"),
        quality_review_count=("requires_quality_review", "sum"),
        quality_review_pct=("requires_quality_review", lambda s: 100 * s.mean()),
    ).round(2)
)

### Score anomalies within each cluster

Fit a separate Isolation Forest for each K-Means cluster. Flag the top 2% of anomaly scores within each cluster, then compare those flags with the existing quality-review indicators.

In [ ]:
from sklearn.ensemble import IsolationForest

anomaly_scores = np.empty(len(X_anomaly), dtype=float)
anomaly_flags = np.zeros(len(X_anomaly), dtype=bool)

for cluster_id in np.unique(anomaly_cluster_labels):
    cluster_idx = np.flatnonzero(anomaly_cluster_labels == cluster_id)
    X_cluster = X_anomaly[cluster_idx]

    detector = IsolationForest(
        n_estimators=300,
        contamination="auto",
        random_state=42,
        n_jobs=-1
    )
    detector.fit(X_cluster)

    scores = -detector.score_samples(X_cluster)
    cutoff = np.quantile(scores, 0.98)

    anomaly_scores[cluster_idx] = scores
    anomaly_flags[cluster_idx] = scores >= cutoff

anomaly_data["isolation_score"] = anomaly_scores
anomaly_data["isolation_flag"] = anomaly_flags
anomaly_data["overlap_with_quality_review"] = (
    anomaly_data["isolation_flag"]
    & anomaly_data["requires_quality_review"].fillna(False)
)

print("Anomaly flags vs. existing quality-review flags:")
display(
    anomaly_data.groupby("kmeans_cluster").agg(
        tour_count=("tour_id", "size"),
        isolation_flagged=("isolation_flag", "sum"),
        quality_review_flagged=("requires_quality_review", "sum"),
        overlap=("overlap_with_quality_review", "sum"),
    )
)

top_anomalies = (
    anomaly_data
    .sort_values(
        ["kmeans_cluster", "isolation_score"],
        ascending=[True, False]
    )
    .groupby("kmeans_cluster")
    .head(5)
)

print("Top five anomaly-ranked tours per cluster:")
display(top_anomalies[
    [
        "vehicle_id",
        "tour_id",
        "kmeans_cluster",
        "isolation_score",
        "total_distance_km",
        "recorded_track_hours",
        "elapsed_span_hours",
        "distance_based_avg_speed_kmh",
        "track_count",
        "requires_quality_review",
        "duration_incomplete",
        "duration_unusable",
    ]
].round(2))

## Review the highest-scoring tours

Show the five most anomalous tours in each cluster along with their timestamps and existing quality indicators. Anomaly scores are ranked within each cluster.

In [ ]:
review_columns = [
    "vehicle_id",
    "tour_id",
    "kmeans_cluster",
    "isolation_score",
    "start_time",
    "last_track_start",
    "end_time",
    "total_distance_km",
    "recorded_track_hours",
    "elapsed_span_hours",
    "distance_based_avg_speed_kmh",
    "track_count",
    "long_haul",
    "requires_quality_review",
    "duration_incomplete",
    "duration_unusable",
]

priority_review = (
    anomaly_data
    .sort_values(
        ["kmeans_cluster", "isolation_score"],
        ascending=[True, False]
    )
    .groupby("kmeans_cluster")
    .head(5)[review_columns]
)

print(priority_review.to_string(index=False))

In [ ]:
audit_df = anomaly_data.copy()

elapsed = pd.to_numeric(audit_df["elapsed_span_hours"], errors="coerce")
recorded = pd.to_numeric(audit_df["recorded_track_hours"], errors="coerce")
distance = pd.to_numeric(audit_df["total_distance_km"], errors="coerce")

audit_df["elapsed_to_recorded_ratio"] = np.where(
    recorded > 0, elapsed / recorded, np.nan
)

# Exploratory review signals, not automatic error rules
audit_df["span_at_least_7_days"] = elapsed >= 168
audit_df["elapsed_recorded_ratio_at_least_10"] = (
    audit_df["elapsed_to_recorded_ratio"] >= 10
)
audit_df["under_1km_with_over_1h_recorded"] = (
    (distance < 1) & (recorded > 1)
)

review_signals = [
    "span_at_least_7_days",
    "elapsed_recorded_ratio_at_least_10",
    "under_1km_with_over_1h_recorded",
]

summary_rows = []
for cluster, group in audit_df.groupby("kmeans_cluster"):
    isolation_flagged = group["isolation_flag"].fillna(False).astype(bool)

    for signal in review_signals:
        signal_flagged = group[signal].fillna(False).astype(bool)
        summary_rows.append({
            "cluster": cluster,
            "review_signal": signal,
            "tours_matching_signal": int(signal_flagged.sum()),
            "isolation_flags_matching_signal": int(
                (signal_flagged & isolation_flagged).sum()
            ),
            "pct_of_isolation_flags": round(
                100 * (signal_flagged & isolation_flagged).sum()
                / max(isolation_flagged.sum(), 1),
                1,
            ),
        })

signal_summary = pd.DataFrame(summary_rows)
display(signal_summary)

In [ ]:
def make_review_reason(row):
    reasons = []

    if row["span_at_least_7_days"]:
        reasons.append("elapsed span >= 7 days")

    if row["elapsed_recorded_ratio_at_least_10"]:
        reasons.append("elapsed/recorded ratio >= 10")

    if row["under_1km_with_over_1h_recorded"]:
        reasons.append("distance < 1 km with > 1 recorded hour")

    if not reasons:
        reasons.append("unusual within-cluster feature combination")

    return " | ".join(reasons)


anomaly_review_queue = audit_df.loc[
    audit_df["isolation_flag"].fillna(False)
].copy()

anomaly_review_queue["review_reason"] = anomaly_review_queue.apply(
    make_review_reason, axis=1
)

reason_summary = (
    anomaly_review_queue
    .groupby(["kmeans_cluster", "review_reason"])
    .agg(
        flagged_tours=("tour_id", "size"),
        existing_quality_review=("requires_quality_review", "sum"),
        median_isolation_score=("isolation_score", "median"),
    )
    .reset_index()
    .sort_values(["kmeans_cluster", "flagged_tours"], ascending=[True, False])
)

display(reason_summary)

In [ ]:
sensitivity_rows = []

for cluster, group in audit_df.groupby("kmeans_cluster"):
    current_flags = group["isolation_flag"].fillna(False).astype(bool)
    quality_flags = (
        group["requires_quality_review"].fillna(False).astype(bool)
    )

    for target_rate in [0.01, 0.02, 0.05]:
        cutoff = group["isolation_score"].quantile(1 - target_rate)
        candidate_flags = group["isolation_score"] >= cutoff
        overlap_with_current = candidate_flags & current_flags

        sensitivity_rows.append({
            "cluster": cluster,
            "target_flag_rate_pct": int(target_rate * 100),
            "actual_flagged_tours": int(candidate_flags.sum()),
            "existing_quality_review_overlap": int(
                (candidate_flags & quality_flags).sum()
            ),
            "overlap_with_current_2pct_flags": int(
                overlap_with_current.sum()
            ),
            "current_2pct_flags_retained_pct": round(
                100 * overlap_with_current.sum() / max(current_flags.sum(), 1),
                1,
            ),
            "score_cutoff": round(float(cutoff), 6),
        })

sensitivity_summary = pd.DataFrame(sensitivity_rows)
display(sensitivity_summary)

In [ ]:
review_queue = audit_df.loc[
    audit_df["isolation_flag"].fillna(False)
].copy()

review_queue["review_reason"] = review_queue.apply(
    make_review_reason, axis=1
)

review_queue["within_cluster_rank"] = (
    review_queue.groupby("kmeans_cluster")["isolation_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

preview_columns = [
    "vehicle_id",
    "tour_id",
    "kmeans_cluster",
    "within_cluster_rank",
    "isolation_score",
    "review_reason",
    "total_distance_km",
    "recorded_track_hours",
    "elapsed_span_hours",
    "distance_based_avg_speed_kmh",
    "track_count",
    "requires_quality_review",
]

print("Tours in the 2% review queue:", len(review_queue))
print(
    review_queue
    .sort_values(["kmeans_cluster", "within_cluster_rank"])
    .groupby("kmeans_cluster", group_keys=False)
    .head(3)[preview_columns]
    .to_string(index=False)
)

In [ ]:
data_dir = repo_root / "data"

if not data_dir.exists():
    print("Data folder not found:", data_dir)
else:
    files = sorted(path for path in data_dir.rglob("*") if path.is_file())

    print("Data files found:", len(files))
    for path in files:
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"{path.relative_to(repo_root)}  ({size_mb:.1f} MB)")

In [ ]:
candidate_paths = {
    "raw tracks": repo_root / "data/raw/tracks/tracks.csv",
    "engineered tracks": repo_root / "data/processed/engineered_tracks.csv",
    "retained tours": repo_root / "data/processed/tour_level/retained_tours.csv",
    "quality-checked tours": (
        repo_root / "data/processed/tour_level/retained_tours_quality_checked.csv"
    ),
}

for label, path in candidate_paths.items():
    if path.exists():
        sample = pd.read_csv(path, nrows=2)
        print(f"\n{label}: {path.relative_to(repo_root)}")
        print("Columns:", list(sample.columns))
        print(sample.to_string(index=False))
    else:
        print(f"\n{label}: not found")

In [ ]:
raw_tracks_path = repo_root / "data/raw/tracks/tracks.csv"

raw_track_data = pd.read_csv(
    raw_tracks_path,
    usecols=[
        "track_id",
        "vehicle_id",
        "tour_id",
        "start_time",
        "stop_time",
        "distance",
    ],
)

raw_track_data["start_utc"] = pd.to_datetime(
    raw_track_data["start_time"], utc=True, errors="coerce"
)
raw_track_data["stop_utc"] = pd.to_datetime(
    raw_track_data["stop_time"], utc=True, errors="coerce"
)

selected_tours = [
    (66, 12776),    # cluster 0: unusual elapsed/recorded ratio
    (58, 10150),    # cluster 1: near-zero distance with recorded time
    (162, 13436),   # cluster 2: very long span
    (131, 13165),   # cluster 3: very long span with few tracks
]

for vehicle_id, tour_id in selected_tours:
    tour_tracks = raw_track_data.loc[
        (raw_track_data["vehicle_id"] == vehicle_id)
        & (raw_track_data["tour_id"] == tour_id)
    ].copy()

    tour_tracks = tour_tracks.sort_values(
        ["start_utc", "stop_utc"]
    ).reset_index(drop=True)

    tour_tracks["gap_before_hours"] = (
        tour_tracks["start_utc"] - tour_tracks["stop_utc"].shift(1)
    ).dt.total_seconds() / 3600

    expected = anomaly_data.loc[
        (anomaly_data["vehicle_id"] == vehicle_id)
        & (anomaly_data["tour_id"] == tour_id)
    ].iloc[0]

    print(
        f"\nVehicle {vehicle_id}, tour {tour_id}, "
        f"cluster {int(expected['kmeans_cluster'])}"
    )
    print(
        f"Expected tracks: {int(expected['track_count'])}; "
        f"raw track rows: {len(tour_tracks)}; "
        f"tour elapsed span: {expected['elapsed_span_hours']:.2f} h"
    )

    gap_rows = tour_tracks.loc[
        tour_tracks["gap_before_hours"].notna()
    ].sort_values("gap_before_hours", ascending=False)

    if gap_rows.empty:
        print("No between-track gaps to calculate.")
    else:
        print("Largest between-track gaps:")
        print(
            gap_rows[
                [
                    "track_id",
                    "start_utc",
                    "stop_utc",
                    "gap_before_hours",
                    "distance",
                ]
            ]
            .head(3)
            .to_string(index=False)
        )

In [ ]:
gap_audit = raw_track_data.sort_values(
    ["vehicle_id", "tour_id", "start_utc", "stop_utc"]
).copy()

gap_audit["gap_before_hours"] = (
    gap_audit["start_utc"]
    - gap_audit.groupby(["vehicle_id", "tour_id"])["stop_utc"].shift(1)
).dt.total_seconds() / 3600

track_gap_summary = (
    gap_audit.groupby(["vehicle_id", "tour_id"])
    .agg(
        raw_track_rows=("track_id", "size"),
        max_intertrack_gap_hours=("gap_before_hours", "max"),
        gaps_over_24h=("gap_before_hours", lambda s: int((s >= 24).sum())),
        gaps_over_7d=("gap_before_hours", lambda s: int((s >= 168).sum())),
    )
    .reset_index()
)

review_queue = review_queue.drop(
    columns=[
        "raw_track_rows",
        "max_intertrack_gap_hours",
        "gaps_over_24h",
        "gaps_over_7d",
    ],
    errors="ignore",
).merge(
    track_gap_summary,
    on=["vehicle_id", "tour_id"],
    how="left",
    validate="one_to_one",
)

review_queue["track_count_matches"] = (
    pd.to_numeric(review_queue["track_count"], errors="coerce")
    == review_queue["raw_track_rows"]
)

elapsed = pd.to_numeric(review_queue["elapsed_span_hours"], errors="coerce")
review_queue["largest_gap_share_pct"] = (
    100 * review_queue["max_intertrack_gap_hours"] / elapsed.where(elapsed > 0)
)

review_queue["has_gap_over_24h"] = (
    review_queue["max_intertrack_gap_hours"] >= 24
)
review_queue["has_gap_over_7d"] = (
    review_queue["max_intertrack_gap_hours"] >= 168
)

cluster_gap_summary = (
    review_queue.groupby("kmeans_cluster")
    .agg(
        flagged_tours=("tour_id", "size"),
        track_count_match_pct=("track_count_matches", "mean"),
        tours_with_gap_over_24h=("has_gap_over_24h", "sum"),
        tours_with_gap_over_7d=("has_gap_over_7d", "sum"),
        median_largest_gap_hours=("max_intertrack_gap_hours", "median"),
        median_largest_gap_share_pct=("largest_gap_share_pct", "median"),
    )
    .reset_index()
)

cluster_gap_summary["track_count_match_pct"] *= 100
display(cluster_gap_summary.round(1))

In [ ]:
gap_hours = pd.to_numeric(
    review_queue["max_intertrack_gap_hours"], errors="coerce"
)

review_queue["gap_category"] = np.select(
    [
        gap_hours.notna() & (gap_hours >= 168),
        gap_hours.notna() & (gap_hours >= 24),
        gap_hours.notna(),
    ],
    [
        "Gap >= 7 days",
        "Gap 24 hours to 7 days",
        "No gap >= 24 hours",
    ],
    default="No measurable inter-track gap",
)

combined_review_summary = (
    review_queue
    .groupby(["kmeans_cluster", "gap_category"], dropna=False)
    .agg(
        flagged_tours=("tour_id", "size"),
        existing_quality_review=("requires_quality_review", "sum"),
        median_isolation_score=("isolation_score", "median"),
        median_largest_gap_hours=("max_intertrack_gap_hours", "median"),
    )
    .reset_index()
    .sort_values(["kmeans_cluster", "flagged_tours"], ascending=[True, False])
)

display(combined_review_summary.round(2))

In [ ]:
queue_columns = [
    "vehicle_id",
    "tour_id",
    "kmeans_cluster",
    "within_cluster_rank",
    "isolation_score",
    "review_reason",
    "gap_category",
    "max_intertrack_gap_hours",
    "largest_gap_share_pct",
    "gaps_over_24h",
    "gaps_over_7d",
    "total_distance_km",
    "recorded_track_hours",
    "elapsed_span_hours",
    "distance_based_avg_speed_kmh",
    "track_count",
    "raw_track_rows",
    "track_count_matches",
    "requires_quality_review",
    "duration_incomplete",
    "duration_unusable",
    "start_time",
    "last_track_start",
    "end_time",
]

review_queue_export = review_queue[queue_columns].copy()
review_queue_export["manual_review_status"] = "pending"
review_queue_export["manual_review_notes"] = ""

output_dir = repo_root / "data/processed/anomaly_detection"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "isolation_forest_review_queue_2pct.csv"
review_queue_export.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Review-queue rows:", len(review_queue_export))
print("\nGap categories:")
print(review_queue_export["gap_category"].value_counts().to_string())

In [ ]:
for relative_path in [
    "raw/tracks/tracks.csv",
    "processed/engineered_tracks.csv",
    "processed/tour_level/retained_tours.csv",
    "processed/tour_level/retained_tours_quality_checked.csv",
]:
    path = repo_root / "data" / relative_path
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"{relative_path} ({size_mb:.1f} MB)")
    else:
        print(f"{relative_path} not found")

## Anomaly Detection Findings

Isolation Forest was applied within each of the four K-Means clusters using the four clustering features: distance, recorded track hours, elapsed span, and track count. The top 2% of anomaly scores within each cluster were selected for review, producing 270 tours.

### Track-gap audit

Raw track counts matched the tour summaries for every flagged tour. Of the 270 candidates:

- 81 have a largest inter-track gap of at least 24 hours.
- 34 have a largest inter-track gap of at least 7 days.
- 160 have a measurable largest gap below 24 hours.
- 29 have no measurable inter-track gap.

Large gaps are concentrated in clusters 2 and 3: 54 of their 66 flagged tours have a gap of at least 24 hours. Cluster 1 has no flagged tours with a gap that large, so its unusual cases need review for other patterns, including near-zero distance despite recorded time.

### Interpretation and limitations

The model identifies tours that are unusual relative to other tours in the same cluster. Its anomaly score is a ranking, not an error probability. The 2% cutoff is a practical review-volume choice, not a validated optimum.

Only 17 of the 270 model flags overlap with existing quality-review flags. This shows that the model surfaces mostly different cases; it does not establish which flags are correct. Long gaps are review signals and do not by themselves prove that a tour is invalid.

The file `data/processed/anomaly_detection/isolation_forest_review_queue_2pct.csv` contains the candidates, reasons, gap diagnostics, and blank fields for manual review.